# EcoSort AI: train the detector on Kaggle

This notebook does what `train_ecosort.ipynb` does in Colab, but it runs from start to finish **without anyone
watching**. You can close the browser or lock your phone. Kaggle gives about 30 free GPU hours a week and
sessions of up to 12 hours; one training run takes about 8 to 9 hours.

## One-time setup

1. **Kaggle account:** sign up at kaggle.com. Under **Settings**, verify your phone number. Kaggle needs it
   before it lets you use a GPU or the internet.
2. **Import this notebook:** on Kaggle, **Create → New Notebook**, then **File → Import Notebook** and give it
   this file's GitHub link:
   `https://github.com/TheAnakin01/ecosort-ai/blob/feature/production-readiness-roadmap/ai-model/notebooks/train_ecosort_kaggle.ipynb`
3. **Notebook settings** (the panel on the right, or **⋮ → Settings** on a phone):
   - **Accelerator:** GPU T4 x2
   - **Internet:** On
4. **Secrets** (**Add-ons → Secrets**). Add these and tick **Attach** for this notebook:
   - `ROBOFLOW_API_KEY`: your Roboflow **Private** API key (Roboflow → Settings → API Keys).
   - `NTFY_TOPIC` (optional): your ntfy topic name, for phone notifications after every epoch and at the end.

## Every run

**Save Version → Save & Run All (Commit) → Save.** Kaggle runs every cell in the background.
When it finishes, open the version and go to **Output**: the trained model is `best.pt`, and the whole run
is in `runs/`.

Before committing, you can run the first cell interactively to check that the GPU, internet and secrets are set up.

## 1. Check the settings

In [ ]:
import csv
import os
import subprocess
import threading
import time
import urllib.request
from pathlib import Path

WORK = Path("/tmp")  # code and data: not kept after the run
OUT = Path("/kaggle/working")  # kept after the run: the version's Output tab
RUNS = OUT / "runs"
STARTED = time.time()

try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
    gpu_names = gpu.stdout.strip().replace("\n", ", ") if gpu.returncode == 0 else ""
except FileNotFoundError:
    gpu_names = ""
if not gpu_names:
    raise SystemExit("No GPU. In the notebook settings set Accelerator to 'GPU T4 x2', then run again.")
print("GPU:", gpu_names)

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
try:
    os.environ["ROBOFLOW_API_KEY"] = secrets.get_secret("ROBOFLOW_API_KEY").strip()
except Exception:
    raise SystemExit(
        "No ROBOFLOW_API_KEY secret. Add-ons -> Secrets -> Add secret, label it ROBOFLOW_API_KEY, "
        "paste the Private API Key, and tick Attach for this notebook."
    ) from None
try:
    TOPIC = secrets.get_secret("NTFY_TOPIC").strip()
except Exception:
    TOPIC = ""
print("Roboflow key: found")
print("Phone notifications:", f"on (ntfy topic {TOPIC})" if TOPIC else "off")


def notify(message, priority="default"):
    """Send a phone notification through ntfy, if a topic is set. Never stops the run."""
    if not TOPIC:
        return
    try:
        request = urllib.request.Request(
            f"https://ntfy.sh/{TOPIC}",
            data=message.encode(),
            headers={"Title": "EcoSort training (Kaggle)", "Priority": priority},
        )
        urllib.request.urlopen(request, timeout=10)
    except Exception as error:
        print("Notification failed:", error)


def run(step, *command):
    """Run one step as its own process and show its output. If it fails, notify and stop."""
    print(f"\n=== {step} ===", flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in process.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line.rstrip()])[-8:]
    if process.wait() != 0:
        hours = (time.time() - STARTED) / 3600
        notify(f"STOPPED at '{step}' after {hours:.1f} h:\n" + "\n".join(tail[-5:]), priority="urgent")
        raise SystemExit(f"'{step}' failed. Read the output above.")


notify("Kaggle run started")

## 2. Get the code and libraries

Every step runs its script as a separate process, so the library upgrades here cannot clash with anything
already loaded in the notebook.

In [ ]:
BRANCH = "feature/production-readiness-roadmap"
REPO_URL = "https://github.com/TheAnakin01/ecosort-ai.git"

if not (WORK / "ecosort-ai").exists():
    run("Get the code", "git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(WORK / "ecosort-ai"))
os.chdir(WORK / "ecosort-ai" / "ai-model")
run("Checkout branch", "git", "checkout", BRANCH)
run("Pull branch", "git", "pull", "origin", BRANCH)
run("Install libraries", "pip", "install", "-q", "-r", "requirements.txt")
run("Code version", "git", "log", "-1", "--format=%h %s")

In [ ]:
import subprocess

EXPECTED_BRANCH = "feature/production-readiness-roadmap"


def git(*args):
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    if result.returncode != 0:
        msg = "git " + " ".join(args) + " failed: " + result.stderr
        raise SystemExit(msg)
    return result.stdout.strip()


current_branch = git("branch", "--show-current")
commit_hash = git("rev-parse", "HEAD")
commit_subject = git("log", "-1", "--oneline")

print("=== ECOSORT AI GIT VERIFICATION ===")
print(f"Branch: {current_branch}")
print(f"Commit: {commit_hash}")
print(f"Subject: {commit_subject}")

if current_branch != EXPECTED_BRANCH:
    msg = "Expected branch " + EXPECTED_BRANCH + ", but got " + current_branch + ". Check the notebook's Git clone settings."
    raise SystemExit(msg)


## 3. Download the four Roboflow datasets

Trash Detection, TrashNet, Neat Now and GARBAGE CLASSIFICATION 3 (the `objectdetection-z2zn1` copy).
The key comes from the `ROBOFLOW_API_KEY` secret. This takes about 10 minutes.

In [ ]:
run("Download datasets", "python", "scripts/download_datasets.py")

## 4. Build and check the merged dataset

The same build that was checked and approved in Colab: four sources, a fresh split that keeps copies of one
photo together, and organic kept to at most 30% of boxes. It prints every class mapping and the box counts,
so the output can be checked afterwards.

The check must find no errors; training will not start otherwise.

In [ ]:
run("Build the dataset", "python", "scripts/prepare_dataset.py", *"downloads/trash-detection downloads/trashnet downloads/neat-now downloads/garbage-cls-3".split(), "--resplit", "--max-share", "organic=0.3")
run("Check the dataset", "python", "scripts/validate_dataset.py")

## 5. Train

30 epochs at most. Training stops early once the validation score has not improved for 10 epochs, so the model
does not overtrain. `best.pt` is always the best epoch, not the last. The run is saved to the Output folder
after every epoch.

With an ntfy topic set, you get a notification after every epoch, and one at the end. If the notifications stop
for more than about 40 minutes, open the notebook's page and check the log.

In [ ]:
def watch_epochs(stop):
    seen = 0
    while not stop.is_set():
        runs = [p for p in RUNS.glob("ecosort-baseline*") if (p / "results.csv").exists()]
        if runs:
            results = max(runs, key=lambda p: p.stat().st_mtime) / "results.csv"
            with results.open(newline="") as f:
                rows = [{k.strip(): v for k, v in row.items()} for row in csv.DictReader(f)]
            if len(rows) > seen:
                seen = len(rows)
                last = rows[-1]
                notify(
                    f"Epoch {seen}/30 done. mAP50 {float(last['metrics/mAP50(B)']):.3f}, "
                    f"mAP50-95 {float(last['metrics/mAP50-95(B)']):.3f}",
                    priority="low",
                )
        stop.wait(120)


stop = threading.Event()
threading.Thread(target=watch_epochs, args=(stop,), daemon=True).start()
try:
    run("Train", "python", "scripts/train.py", "--epochs", "30", "--name", "ecosort-baseline", "--project", str(RUNS))
finally:
    stop.set()

## 6. Check for overtraining and false detections

**Overtraining:** `OVERTRAINING` means validation loss rose while training loss kept falling.

**False detections:** the model is run on the validation photos with the app's settings (confidence 0.35), and
every box is checked against the labels. The **nothing** column counts boxes where there is no waste at all.
The photos with the most of them are shown below and saved in the run folder: red boxes are on nothing,
yellow are other mistakes, green are correct, and white outlines are the labels.

In [ ]:
import shutil

from IPython.display import Image, display

run_dir = max(RUNS.glob("ecosort-baseline*"), key=lambda p: p.stat().st_mtime)
for name in ("dataset_report.json", "validation_report.json"):
    shutil.copy2(Path("datasets/ecosort") / name, run_dir / name)
shutil.copy2(run_dir / "weights" / "best.pt", OUT / "best.pt")

run("Overtraining check", "python", "scripts/overfit_check.py", str(run_dir))
check = run_dir / "hallucination_check_val"
run(
    "False-detection check",
    "python", "scripts/check_hallucinations.py", "--weights", str(run_dir / "weights" / "best.pt"), "--out", str(check),
)

import json

verdict = json.loads((run_dir / "overfit_check.json").read_text())
report = json.loads((check / "report.json").read_text())
hours = (time.time() - STARTED) / 3600
notify(
    f"Finished in {hours:.1f} h. Best mAP50-95 {verdict['best_map50_95']:.3f} at epoch {verdict['best_epoch']}. "
    f"Overtraining check: {verdict['verdict']}. Boxes on nothing: {report['share_on_nothing']:.1%}. "
    "Download best.pt from the Output tab.",
    priority="high",
)

for name in ("results.png", "confusion_matrix.png"):
    if (run_dir / name).exists():
        display(Image(str(run_dir / name), width=900))
for photo in sorted((check / "worst_photos").glob("*"))[:6]:
    display(Image(str(photo), width=600))
print("Trained model:", OUT / "best.pt")

## 7. Next steps

- Download `best.pt` and the `runs/` folder from the version's **Output** tab.
- Copy the validation table, the overtraining verdict and the false-detection table into `docs/training-log.md`.
- The backend serves ONNX: convert with `python scripts/export_onnx.py --weights best.pt` before deploying.
- Do not run a test-split evaluation yet. That is the final score, run once, after the confidence threshold
  has been chosen on the validation split.